In [ ]:
#You need Mediapipe 0.10.14
# pip install mediapipe==0.10.14
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils #to draw
mp_pose = mp.solutions.pose #pose estimation model

In [2]:
#VIDEO FEED
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    cv2.imshow("Mediapipe Feed", frame)
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
    
cap.release()
cv2.destroyAllWindows()

1. Make detections

In [3]:
cap = cv2.VideoCapture(0)
## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:    
    while cap.isOpened():
        ret, frame = cap.read()
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        
        #Make detection
        results = pose.process(image)
        
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                  )
        
        cv2.imshow("Mediapipe Feed", image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    
cap.release()
cv2.destroyAllWindows()

c:\Users\andrea\Documents\Corso Nearable\Human Pose\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


2. Determining Joints landmarks

In [4]:
cap = cv2.VideoCapture(0)
## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:    
    while cap.isOpened():
        ret, frame = cap.read()
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        
        # Make detection
        results = pose.process(image)
        
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            print(landmarks)
        except:
            pass
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                  )
        
        cv2.imshow("Mediapipe Feed", image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    
cap.release()
cv2.destroyAllWindows()

[x: 0.676892757
y: 0.805161774
z: -1.6079185
visibility: 0.999875426
, x: 0.697410583
y: 0.73354286
z: -1.56178927
visibility: 0.999838233
, x: 0.714332581
y: 0.728555322
z: -1.56162786
visibility: 0.999880433
, x: 0.730071187
y: 0.72472012
z: -1.56167877
visibility: 0.999767482
, x: 0.640672445
y: 0.742327273
z: -1.5658735
visibility: 0.999832392
, x: 0.620615304
y: 0.742430508
z: -1.5653652
visibility: 0.999884248
, x: 0.600797474
y: 0.74302727
z: -1.56545496
visibility: 0.999834895
, x: 0.755980611
y: 0.731999099
z: -1.1039511
visibility: 0.999902129
, x: 0.573363602
y: 0.76289016
z: -1.11055779
visibility: 0.999945045
, x: 0.711521804
y: 0.865488172
z: -1.41746318
visibility: 0.999827862
, x: 0.646370649
y: 0.876801908
z: -1.42229903
visibility: 0.999874234
, x: 0.906156182
y: 1.00156665
z: -0.705221057
visibility: 0.984493792
, x: 0.453127712
y: 0.995114863
z: -0.709641635
visibility: 0.994975567
, x: 1.05425739
y: 1.31106758
z: -0.959344089
visibility: 0.0922857523
, x: 0.3942127

In [5]:
len(landmarks)

33

In [6]:
for landmark in mp_pose.PoseLandmark:
    print(landmark)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32


In [7]:
landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]

x: 0.872694314
y: 1.0134989
z: -0.145700574
visibility: 0.90954268

In [8]:
landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value]

x: 1.02539539
y: 1.29620564
z: -0.106051251
visibility: 0.393308342

In [9]:
landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]

x: 0.92775774
y: 1.05741513
z: -0.238707498
visibility: 0.481734335

3. Calculate Angles

In [10]:
def calculate_angle(a,b,c):
    a = np.array(a) # First
    b = np.array(b) # Mid
    c = np.array(c) # End
    
    
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0]) 
    angle = np.abs(radians * 180.0/np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
        
    return angle

In [11]:
shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].y]
elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW].y]
wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y]

In [12]:
shoulder, elbow, wrist

([0.8726943135261536, 1.0134989023208618],
 [1.025395393371582, 1.2962056398391724],
 [0.927757740020752, 1.0574151277542114])

In [13]:
calculate_angle(shoulder, elbow, wrist)

np.float64(6.13636044190816)

4. Calculating Angles

In [14]:
cap = cv2.VideoCapture(0)
## Setup mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:    
    while cap.isOpened():
        ret, frame = cap.read()
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        
        # Make detection
        results = pose.process(image)
        
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            
            #Get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].y]
            elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW].y]
            wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y]
            
            # Calculate angle
            angle = calculate_angle(shoulder, elbow, wrist)
            
            #height, width, _ = image.shape
            # VIsualize angle
            cv2.putText(image, str(angle),
                        tuple(np.multiply(elbow, [640, 480]).astype(int)),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                        )
            print(landmarks)
        except:
            pass
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                  )
        
        cv2.imshow("Mediapipe Feed", image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    
cap.release()
cv2.destroyAllWindows()

[x: 0.628362536
y: 0.7076267
z: -0.944329858
visibility: 0.999750912
, x: 0.651272595
y: 0.642034292
z: -0.893733859
visibility: 0.999395251
, x: 0.666267693
y: 0.639899373
z: -0.893725574
visibility: 0.999582
, x: 0.681931317
y: 0.638649344
z: -0.894178212
visibility: 0.999370754
, x: 0.598073483
y: 0.650406241
z: -0.89124763
visibility: 0.999356091
, x: 0.580431938
y: 0.652907729
z: -0.89056474
visibility: 0.999575198
, x: 0.563143671
y: 0.656945467
z: -0.890952706
visibility: 0.999444306
, x: 0.707113802
y: 0.657971263
z: -0.515132368
visibility: 0.999511361
, x: 0.539161742
y: 0.678512931
z: -0.487872124
visibility: 0.999638557
, x: 0.66245991
y: 0.760371923
z: -0.796210468
visibility: 0.999722302
, x: 0.597306967
y: 0.768701792
z: -0.789084
visibility: 0.999733031
, x: 0.865407825
y: 0.970336258
z: -0.255488873
visibility: 0.996940494
, x: 0.421255052
y: 0.973478138
z: -0.302570164
visibility: 0.998865843
, x: 0.965977788
y: 1.32289696
z: -0.248906806
visibility: 0.0801435336
, x:

4. Curl Counter

In [15]:
cap = cv2.VideoCapture(0)

# Curl counter variables
counter = 0
stage = None

## Setup Mediapipe instance
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:    
    while cap.isOpened():
        ret, frame = cap.read()
        
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        
        # Make detection
        results = pose.process(image)
        
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        
        # Extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            
            # Get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].x, landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER].y]
            elbow = [landmarks[mp_pose.PoseLandmark.LEFT_ELBOW].x, landmarks[mp_pose.PoseLandmark.LEFT_ELBOW].y]
            wrist = [landmarks[mp_pose.PoseLandmark.LEFT_WRIST].x, landmarks[mp_pose.PoseLandmark.LEFT_WRIST].y]
            
            # Calculate angle
            angle = calculate_angle(shoulder, elbow, wrist)
            
            # Height, width, _ = image.shape
            # Visualize angle
            cv2.putText(image, str(angle),
                        tuple(np.multiply(elbow, [640, 480]).astype(int)),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                        )
            
            # Curl counter Logic
            if angle > 160:
                stage = "down"
            if angle < 30 and stage == "down":
                stage = "up"
                counter += 1
                print(counter)
                
        except:
            pass
        
        # Render curl counter
        # Setup status box
        
        cv2.rectangle(image, (0,0), (225, 73), (245, 117, 16), -1)
        
        # Rep data
        cv2.putText(image, 'REPS', (15,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), (10,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)
        
        # Stage data
        cv2.putText(image, 'STAGE', (65,12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, stage, (60,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA) 
        
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                                  )
        
        cv2.imshow("Mediapipe Feed", image)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    
cap.release()
cv2.destroyAllWindows()

1
2
3
4
5
